# 03. Serving the fine-tuned model with Ray Serve

**AI Drug Discovery & Development Summit 2026 · Hands-on workshop · Part 3 of 3**

**Workshop series:** [01 Ray and offline inference](01_ray_and_offline_inference.ipynb) · [02 Train and tune](02_train_and_tune.ipynb) · [03 Serve](03_serve.ipynb)

Notebook 02 produced a fine-tuned checkpoint. This notebook puts it behind an endpoint a chemist can use:
the 3D results viewer, prediction from Python, and swapping checkpoints without breaking what is already
in use.

| § | Topic | Ray library | Min |
|---|---|---|---|
| 0 | Setup | | 2 |
| 6 | Serving the fine-tuned model | Ray Serve | 17 |
| 7 | Summary and next steps | | 5 |

Section numbers follow the workshop as a whole, so this notebook starts at Section 6.


## 0. Setup

- The Workspace runs the `aiddd-boltz2-of3` image: Ray, Boltz-2 and 2 OpenFold3 versions (0.5 and 0.4.3).
- This notebook runs on a CPU-only head node. L4 GPU nodes join when a cell requests a GPU and leave when idle.
- Data, model weights and all benchmark results live on shared storage.

In [34]:
import os, sys, json, time, warnings, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import ray

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)

from aiddd_pipeline import (SHARED, TARGETS, BOLTZ_CACHE, OF3_OB_CKPT, OF3_P2_CKPT, OF3_FT_CKPT, OF3_LEGACY,
                            Boltz2Predictor, OF3Predictor, load_ligands, validate, make_scorer)
import workshop_helpers as wh

# Data and weights are downloaded once, ahead of the workshop (README step 2)
needed = [TARGETS[t].path("metadata") for t in wh.TARGET_KEYS] + [BOLTZ_CACHE, OF3_OB_CKPT, OF3_P2_CKPT, OF3_FT_CKPT]
missing = [str(p) for p in needed if not p.exists()]
assert not missing, f"Missing {missing}. Run `python scripts/setup_data.py` and `python scripts/prepare_mpro.py` first."

# True: run every GPU step live (the presenter, or anyone with their own GPUs).
# False: GPU steps load stored results. Serving is notebook 03.
USE_GPUS = False
SERVICE_URL = os.environ.get("AIDDD_SERVICE_URL", "")   # the deployed prediction service used in notebook 03
GPU = "L4"            # GPU type for every GPU step in this notebook

wh.MINE.mkdir(parents=True, exist_ok=True)
print(f"shared data:  {SHARED}")
print(f"your outputs: {wh.MINE}")

try:
    ray.init(runtime_env={"env_vars": {"AIDDD_SHARED": str(SHARED)}}, ignore_reinit_error=True, logging_level="ERROR")
except ValueError:   # inside an Anyscale Job, which already sets the environment
    ray.init(ignore_reinit_error=True, logging_level="ERROR")
dashboard = os.environ.get("RAY_OVERRIDE_DASHBOARD_URL") or ray.get_runtime_context().dashboard_url
print(f"Ray Dashboard metrics: {dashboard.rstrip('/')}/#/metrics   (open in a new browser tab)")

shared data:  /mnt/user_storage/aiddd
your outputs: /mnt/user_storage/aiddd/workshop/alicia_chua_30e1ffa
Ray Dashboard metrics: https://session-thqdxs9cduqis7hsp7r4mvd42v.i.anyscaleuserdata.com/#/metrics   (open in a new browser tab)


In [ ]:
# Context from notebooks 01 and 02, loaded here so this notebook stands alone
results = {t: wh.load_results(t) for t in wh.TARGET_KEYS}
ev71 = results["ev71-2a"]
x7259a = ev71[ev71.complex_name == "A71EV2A-x7259a"].iloc[0]     # the late follow-on used in 6.3 to 6.5
print("loaded benchmark results:", ", ".join(results))


## 6. Serving with Ray Serve: one base model, many specialists

⏱ **Budget: 17 minutes.**

A chemist does not want a different endpoint per model. Ray Serve **model multiplexing** serves the
general **base model** and every program's **specialist** from one pool of GPU replicas, chosen per
request, and loads each checkpoint on demand. This section runs that pool in the Workspace and walks the
lifecycle: start with the base model, predict (it is weak on EV-A71 2A), add the program's specialist,
then swap in a newer version.

### 6.1 Start the pool with the base model

`@serve.multiplexed` is the whole mechanism: `load` reads the checkpoint for a `model_id`, and Ray caches
it per id and evicts the least-used past `max_num_models_per_replica`. `__call__` reads the id a request
asked for with `serve.get_multiplexed_model_id()`. We start the pool serving one model, `base`.

In [ ]:
from ray import serve
from aiddd_pipeline import run_openfold_pose      # checkpoint + SMILES -> predicted pose (CIF path)

assert USE_GPUS, "Section 6 starts a GPU pool with serve.run; set USE_GPUS = True (needs an L4)."


@serve.deployment(ray_actor_options={"num_gpus": 0.5, "accelerator_type": "L4"})
class PoseModel:
    def reconfigure(self, checkpoints):                      # {model_id: checkpoint path}
        self.checkpoints = checkpoints

    @serve.multiplexed(max_num_models_per_replica=3)         # keep 3 checkpoints hot, evict the rest
    async def load(self, model_id):
        return self.checkpoints[model_id]                    # the model is just a checkpoint path

    async def __call__(self, smiles):
        checkpoint = await self.load(serve.get_multiplexed_model_id())
        return run_openfold_pose(checkpoint, smiles)


models = {"base": str(OF3_P2_CKPT)}
handle = serve.run(PoseModel.options(user_config=models).bind(), name="aiddd-pose")

### 6.2 Compounds for the demo

All are EV-A71 2A late follow-ons with a crystal structure, so you can compare the prediction with the
experiment.

| Compound | Base model | Our fine-tune (v1) | Public fine-tune (v2) | SMILES |
|---|---|---|---|---|
| **x7259a**, the demo: most potent here (pKD 7.9) | 5.1 Å | 1.0 Å | 0.4 Å | `CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7101a**, harder: only v2 gets it | 6.2 Å | 5.1 Å | 0.8 Å | `CSc1ncnc(N2CCC3(CCCCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |

Ligand RMSD to the crystal; under 2 Å is a correct pose. One diffusion sample, seed 42, as this
notebook runs them.

### 6.3 Predict with the base model

`handle.options(multiplexed_model_id="base")` is the one line that picks a model per request. The base
model has never seen this pocket with a ligand in it, so on an EV-A71 2A follow-on it is usually wrong.

In [ ]:
SMILES = "CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1"     # x7259a
crystal = x7259a

cif = await handle.options(multiplexed_model_id="base").remote(SMILES)
wh.show_analog(cif, crystal, label="x7259a (base model)", height=340)

### 6.4 Add the program's specialist

Notebook 02 fine-tuned the base model on this program's fragment screen. Add its checkpoint under a new
id and re-run `serve.run`: the base model keeps serving everyone else, and the specialist loads on the
first request for its id.

In [ ]:
from aiddd_pipeline import OF3_OURS_CKPT
assert OF3_OURS_CKPT.exists(), f"Missing {OF3_OURS_CKPT}: copy the fine-tune run to shared storage first."
models["ev71-v1"] = str(OF3_OURS_CKPT)     # our fine-tune: 1,000 steps on 8 L4s
handle = serve.run(PoseModel.options(user_config=models).bind(), name="aiddd-pose")

cif = await handle.options(multiplexed_model_id="ev71-v1").remote(SMILES)
wh.show_analog(cif, crystal, label="x7259a (specialist v1)", height=340)

Same molecule, same endpoint, a different `multiplexed_model_id`: the pose moves from 5.1 Å off the
crystal to 1.0 Å. The base model and the specialist share the GPU replica.

### 6.5 Swap in a newer version

A stronger fine-tune arrives (here, the OpenBind release: 0.4 Å on x7259a, and it also gets x7101a). Add it and drop the first specialist: for this
program, the new version takes its place. (Multiplexing would also let you keep both ids for a canary or
an instant rollback.)

In [ ]:
models["ev71-v2"] = str(OF3_FT_CKPT)     # the OpenBind release
del models["ev71-v1"]                     # the new version replaces the first specialist
handle = serve.run(PoseModel.options(user_config=models).bind(), name="aiddd-pose")

cif = await handle.options(multiplexed_model_id="ev71-v2").remote(SMILES)
wh.show_analog(cif, crystal, label="x7259a (specialist v2)", height=340)

**On an Anyscale Service.** The workshop Service (`jobs/service.yaml`, app `serve_viewer.py`) runs the same
lifecycle behind the web viewer. Each step changes only `user_config`, so `--in-place` reconfigures the
running replicas: no restart, no new nodes. The viewer lists each version's answer for the same molecule.

```bash
anyscale service deploy -f jobs/service.yaml                     # once, ahead of time (5 to 10 min);
                                                                 # add --in-place if it is already running
anyscale service deploy -f jobs/demo_1_base.yaml --in-place       # base model only
anyscale service deploy -f jobs/demo_2_ft.yaml --in-place         # add our fine-tune, v1
anyscale service deploy -f jobs/demo_3_next_ft.yaml --in-place    # swap in v2
```

Edit no file in the repo between the first deploy and the steps: the upload would then carry a code
change, and each step would restart every replica instead of reconfiguring it. Notebooks are not
uploaded, so running and editing this one is safe.

A change to code or the image deploys without `--in-place` and can shift traffic gradually:
`--canary-percent 10`, then `--canary-percent 100` to promote, or `anyscale service rollback -n aiddd-viewer`.

| What Ray Serve and Anyscale add | Why it matters |
|---|---|
| **Model multiplexing** | The base model and every specialist on the same GPUs, chosen per request |
| **On-demand loading** | A new model is served by requesting its id; no redeploy |
| **GPUs from every zone** | When one zone runs out of L4s, replicas start in another |
| **Canary rollouts and rollback** | A new model reaches a share of users first |

## 7. Summary and next steps

**Findings**

- **Generalization.** Models that reach 88 to 92% on SARS-CoV-2 late follow-ons fall to 1 to 24% on EV-A71 2A.
- **Fine-tuning.** 66 fragment-screen structures measurably improve OpenFold3 on held-out follow-on
  compounds (notebook 02), and the result is a specialist for that target.

**Delivered on a single Ray cluster**

- **Ray Data.** The full benchmark across 3 campaigns, and 402 new OpenFold3 predictions in 19 minutes on
  12 L4 GPUs.
- **Ray Train.** OpenFold3's own training code on 8 GPUs, with checkpoints and automatic recovery.
- **Ray Serve.** The base model and program specialists behind one multiplexed endpoint, chosen per
  request, with on-demand loading and canary rollouts.

The [README](README.md) has the scientific background and the full runbook. Run the next cell to stop the
pool and release the GPU nodes.

In [ ]:
serve.shutdown()
print("Pool stopped. Idle GPU nodes will be released within a few minutes.")